
---

# 论文：《NFT: Bridging Supervised Learning and Reinforcement Learning in Math Reasoning》

本篇论文的核心贡献在于打破了“大语言模型自我提升（Self-Improvement）必须依赖强化学习（RL）”的传统迷信。它提出了一种纯监督学习（SL）框架——**负样本感知微调（Negative-aware Fine-Tuning, NFT）**。通过精妙的**全概率公式线性拆分**与**隐式参数化设计**，NFT 在不需要 Critic 网络和复杂的优势归一化代码的情况下，在数学推理任务上达到甚至超越了 GRPO 等主流 RL 算法。

---

## 一、 核心动机：SL 过去输给 RL 的真正原因

在数学推理任务中，输入一个数学题 $q$，模型生成解答 $a$。通过一个外部验证器（如 Python 解释器或规则匹配）来获得二元奖励：$r(q,a) \in \{0, 1\}$（1 表示正确，0 表示错误）。

* **传统监督学习（拒绝采样微调，RFT）：** 直接丢弃所有错题（$r=0$），只把对的题（$r=1$）收集起来做最大似然估计（MLE）。
* **强化学习（如 GRPO）：** 同时利用对的和错的样本。通过在同一组题目内计算相对奖励，若回答错误，该路径的概率会被大力打压。

**论文的关键洞察**：RFT 在在线迭代中不如 RL，**并不是因为监督学习范式本身不行，而是因为它直接把负样本（错题）扔掉了，没有从中吸取教训**。NFT 的目标，就是让监督学习也“看懂”负样本，并以一种数学上最优雅的线性减法结构来完成反思。

---

## 二、 核心理论来源与数学推导

NFT 的精妙之处在于它通过**策略拆分（Policy Splitting）**，隐式地构建了一个“负样本策略”，从而能够用纯监督学习的极大似然损失去优化原本属于强化学习的问题。

### 1. 问题设定与策略拆分（Policy Splitting）

根据全概率公式，旧大模型 $\pi_{\text{old}}(a|q)$ 生成任意回答的概率，可以被天然地拆分为**正确空间**和**错误空间**的条件概率线性组合：

$$\pi_{\text{old}}(a|q) = p_{\text{old}}(a, r=1|q) + p_{\text{old}}(a, r=0|q)$$

依据条件概率展开：


$$\pi_{\text{old}}(a|q) = p_{\text{old}}(r=1|q) \cdot p_{\text{old}}(a|q, r=1) + p_{\text{old}}(r=0|q) \cdot p_{\text{old}}(a|q, r=0)$$

此时，我们严格对齐论文中的物理量符号：

* $r_q \triangleq p_{\text{old}}(r=1|q)$：旧模型面对该题时的**综合正确率（胜率）**。
* $\pi^+(a|q) \triangleq p_{\text{old}}(a|q, r=1)$：**理想正策略**（做对空间）。
* $\pi^-(a|q) \triangleq p_{\text{old}}(a|q, r=0)$：**理想负策略**（做错空间）。

代入后，完美恢复出论文的**公式 (7)**：


$$\pi_{\text{old}}(a|q) = r_q \pi^+(a|q) + (1 - r_q) \pi^-(a|q)$$

### 2. 隐式负策略的参数化设计与全概率守恒证明

为了能够对错题进行梯度优化，我们需要将公式 (7) 中的负策略单独隔离出来：


$$\pi^-(a|q) = \frac{\pi_{\text{old}}(a|q) - r_q \pi^+(a|q)}{1 - r_q}$$

然而，我们的终极目标是得到一个强大的正策略 $\pi^+$，而不是去训练一个独立的“错题专家”。为此，作者展现了极为高明的设计：**直接用正在训练中的目标正策略 $\pi^+_\theta$ 来显式定义（`:=`）出隐式的参数化负策略 $\pi^-_\theta$：**

$$\pi^-_\theta(a|q) := \frac{\pi_{\text{old}}(a|q) - r_q \pi^+_\theta(a|q)}{1 - r_q}$$

#### 终点处的合法性裁判（Theorem 3.1 证明）

这个强行定义的公式在训练未收敛时显然不满足全概率公式。它的合法性是由**最优解处的不动点守恒**来背书的。

当模型对隐式负策略喂入负样本执行最大似然估计（MLE）时，其全局优化目标为：


$$\max_{\theta} \mathbb{E}_{q \sim p(q), a \sim \pi^-(a|q)} \left[ \log \pi^-_\theta(a|q) \right]$$

在模型容量无限的假设下，该 MLE 的全局最优解 $\theta^*$ 必使得参数化分布与真实数据分布完全重合：


$$\pi^-_{\theta^*}(a|q) \equiv \pi^-(a|q)$$

展开各自的代数原型：


$$\frac{\pi_{\text{old}}(a|q) - r_q \pi^+_{\theta^*}(a|q)}{1 - r_q} = \frac{\pi_{\text{old}}(a|q) - r_q \pi^+(a|q)}{1 - r_q}$$

两边同乘 $(1 - r_q)$ 并移项消去 $\pi_{\text{old}}(a|q)$：


$$-r_q \pi^+_{\theta^*}(a|q) = -r_q \pi^+(a|q) \implies \pi^+_{\theta^*}(a|q) = \pi^+(a|q)$$

**结论：** 在负策略得到最优解（$\pi^-_{\theta^*} = \pi^-$）的瞬间，用来表达它的目标正策略也同时被逼到了真正的最优解（$\pi^+_{\theta^*} = \pi^+$）。此时将最优解代回，**完美、严丝合缝地回归并满足了最初的全概率物理守恒公式：**


$$\pi_{\text{old}}(a|q) = r_q \pi^+_{\theta^*}(a|q) + [1 - r_q] \pi^-_{\theta^*}(a|q)$$

### 3. NFT Loss 函数的导出（公式 9）

为了确保训练起点时总损失平稳，论文采用 **Likelihood Ratio（似然比）** 叙事，即对绝对对数似然减去一个与 $\theta$ 无关的旧模型对数似然作为基线。

#### 负样本损失项（$r=0$）的推导：

我们将负样本的绝对损失 $- \log \pi^-_\theta(a|q)$ 减去基线 $- \log \pi_{\text{old}}^-(a|q)$：


$$\mathcal{L}^-(\theta) = - \log \left( \frac{\pi^-_\theta(a|q)}{\pi_{\text{old}}^-(a|q)} \right)$$

1. **分子代入**：$\pi^-_\theta(a|q) = \frac{\pi_{\text{old}}(a|q) - r_q \pi^+_\theta(a|q)}{1 - r_q}$
2. **分母代入**：在训练起点时刻，尚未完全剥离的旧负策略在数值上直接近似等于大模型未更新前的总似然：$\pi_{\text{old}}^-(a|q) = \pi_{\text{old}}(a|q)$。

两式相除，并将分母的 $\pi_{\text{old}}(a|q)$ 强行分配进分子括号内部：


$$\frac{\pi^-_\theta(a|q)}{\pi_{\text{old}}^-(a|q)} = \frac{\frac{\pi_{\text{old}}(a|q) - r_q \pi^+_\theta(a|q)}{1 - r_q}}{\pi_{\text{old}}(a|q)} = \frac{1 - r_q \frac{\pi^+_\theta(a|q)}{\pi_{\text{old}}(a|q)}}{1 - r_q}$$

#### 最终总损失函数（公式 9）：

加上正样本（$r=1$）的标准监督似然比损失 $- \log \frac{\pi^+_\theta(a|q)}{\pi_{\text{old}}(a|q)}$，用二元指针 $r$ 做开关，达成**论文公式 (9) 的终极形态**：


$$\mathcal{L}_{\text{NFT}}(\theta) = r \left[ - \log \frac{\pi^+_\theta(a|q)}{\pi_{\text{old}}(a|q)} \right] + (1 - r) \left[ - \log \frac{1 - r_q \frac{\pi^+_\theta(a|q)}{\pi_{\text{old}}(a|q)}}{1 - r_q} \right]$$

### 4. 工业级完全体 Loss（公式 10）

$$\mathcal{L}_{\mathcal{D}}^{\text{NFT}}(\theta) = - \sum_{q, a, r} \omega(q) \sum_{t} \left[ r \log R_\theta^t(q, a) + (1 - r) \log \text{max\_v}\left( \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right) \right]$$

其中，**Token 级似然比**与**经验胜率**定义为：


$$R_\theta^t(q, a) = \frac{\pi_\theta(a_t | q, a_{<t})}{\pi_{\text{old}}(a_t | q, a_{<t})} \quad , \quad \hat{r}_q = \frac{1}{K} \sum_{a|q} r(q, a)$$

论文作者为了让公式 10 在大模型长思维链（CoT）训练中稳定落地，焊接了以下**三大工程支柱**：

---

#### 支柱一：Token 级别的损耗解耦（Token-level Loss）

* **理想公式（公式 9）的死穴**：
公式 9 采用的是句子级似然 $\pi(a|q) = \prod_t \pi(a_t|q, a_{<t})$。在长推理（CoT）任务中，一条样本动辄 1024 甚至几千个 Token。连乘会导致序列概率极度稀释，带来恐怖的**梯度估计高方差（High Variance）**。长度越长的回答，更新幅度越畸形，导致训练极度不稳定。
* **工业级改造**：
仿照 PPO/GRPO 等强化学习的常规操作，NFT 将整个句子拆散，**将每一个 Token 的决策视作一个独立的智能体单元**。外层对 $t$ 进行求和（$\sum_t$），将句子级连乘打散为 Token 级连加。这直接抹平了由于文本长度带来的方差不对称，也是工业界大模型微调能收敛的基础。

---

#### 支柱二：负样本对数防爆阀（Clipping & STE）

这是整个 NFT 代码最容易产生 NaN 的地方，也是作者最精妙的工程设计。

* **致命的数学危机**：
看错题（$r=0$）对应的对数项内部：$\frac{1 - \hat{r}_q R_\theta^t}{1 - \hat{r}_q}$。
在训练初期，模型参数 $\theta$ 极不稳定。如果在某一个错题 Token 上，当前模型算出的概率远大于旧模型，导致 $R_\theta^t$ 冲得太高，使得：

$$1 - \hat{r}_q R_\theta^t \le 0$$



那么对数函数 $\log(\le 0)$ 会**瞬间触发数学下溢，吐出 NaN，直接导致整台集群的训练塌方**。
* **防爆阀设计：`max_v(·, ε)**`：
作者强行引入了一个截断算子，通过 $\text{max\_v}(\cdot, \epsilon)$ 确保对数内部的输入永远大于一个极小的正数 $\epsilon > 0$（通常为 `1e-8`）。
* **无痛通关：直通梯度估算（Straight-Through Estimator, STE）**：
常规的 `torch.clamp` 或 `max` 操作在触发截断时，其数学导数会**直接变成 0**。一旦导数变成 0，这部分 Token 的梯度流就断了，模型就失去了从这道错题中反思的能力。
为了解决这个“断流”问题，作者引入了 **STE（Bengio et al., 2013）**。它的核心逻辑是：
* **Forward（前向传播）**：正常执行截断，低于 $\epsilon$ 的统统变成 $\epsilon$，防止 $\log$ 爆炸。
* **Backward（反向传播）**：假装截断没有发生，把后面的梯度**原封不动地跨过 `max_v` 传回去**。


通过这种“前向拦截，后向直通”的硬核魔改，既保住了服务器不炸，又保住了反向擦除错误路径的梯度流。

---

#### 支柱三：提示词动态难度加权（Prompt Weighting $\omega(q)$）

* **为什么需要 $\omega(q)$？**
在复杂的数学题库中，题目的难度是天差地别的。如果不对题目做加权，模型会花大量的时间去反复刷那些已经能轻松做对的“简单题”，而在真正能提供丰富信息量的“硬核错题”上训练不足。
* **权重分配逻辑**：
公式中的 $\omega(q)$ 是关于当前模型对该题胜率 $\hat{r}_q$ 的函数。它会**给低 $\hat{r}_q$（胜率低、难度高）的 Prompt 赋予更高的 Loss 权重**。
* **与 GRPO 的暗中契合**：
这不仅提升了训练效率，更是 NFT 在多道题目的宏观层面（Inter-prompt）与 GRPO 达成对齐的纽带。GRPO 通过群体相对优势天然拉平了题目难度，而 NFT 在隐式状态下必须靠 $\omega(q)$ 这个外部补丁，才能在宏观梯度分布上与 GRPO 达成等价。

---

#### 重新审视：完全体 NFT 的工程优势

现在把公式 10 的完全体和它的三大工程防爆设计拉通来看，它才真正具备了单 Policy 模型单挑强化学习框架的实力：

| 维度 | 理论级 NFT（公式 9） | 工业级 NFT（公式 10） |
| --- | --- | --- |
| **基本更新单元** | 句子级别（高方差，长文本易崩） | **Token 级别**（方差稳定，完美适配 CoT） |
| **数值安全保障** | 无（数值无边界，极易遭遇 $\log(\le 0) \to \text{NaN}$） | **`max_v` + STE 直通截断**（既防爆，又不断梯度） |
| **宏观题目均衡** | 题目一视同仁（容易在简单题上发生梯度空转） | **难度加权 $\omega(q)$**（聚焦高价值硬核错题） |

有了这一层 Token 级拆解和 STE 截断，NFT 才真正从黑板上的代数秀，变成了能在数万张 GPU 上和 GRPO 正面硬刚的工业级算法。


---

## 三、 NFT 与 GRPO 算法的等价性证明

在这里，作者通过极其严密的**解析梯度比对（Gradient Comparison）**，撕开了强化学习（RL）与监督学习（SL）的表面隔阂，向世人证明：**在同策略（On-Policy）状态下，NFT 与 GRPO 在数学上完全等价；而它们唯一的区别，仅仅在于异策略（Off-Policy）时的梯度截断机制。**

---

### 1. 梯度对决：Proposition 4.1 的精确数学解析

为了探究两者的本质差异，论文设定了一个极其严格的单题控制变量实验：

* 设在当前问题 $q$ 下，模型通过同策略 Rollout 采样的总样本数为 $K$。
* 其中，做对的正样本数量为 $\hat{r}_q K$，做错的负样本数量为 $(1 - \hat{r}_q) K$。此时 $\hat{r}_q$ 为该题的经验胜率。

#### a. GRPO 的梯度形态

在二元奖励 $r \in \{0, 1\}$ 下，标准的 GRPO 算法对 Token 级似然比 $R_\theta^t(q, a)$ 求导，其解析梯度公式为：

$$\nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{GRPO}}(\theta) = - \sum_{q,a,r} \left\{ r A_q^+ \cdot \mathcal{I}[R_\theta^t(q,a) < 1 + \epsilon'] + (1-r) A_q^- \cdot \mathcal{I}[R_\theta^t(q,a) > 1 - \epsilon'] \right\} \nabla_\theta R_\theta^t(q,a)$$

其中，**$A_q^+$ 与 $A_q^-$ 是经过群体归一化（Group Normalization）后得到的标准优势值**：


$$A_q^+ = \sqrt{\frac{1-\hat{r}_q}{\hat{r}_q}} \quad , \quad A_q^- = -\sqrt{\frac{\hat{r}_q}{1-\hat{r}_q}}$$

* $\mathcal{I}[\cdot]$ 为指示函数（Indicator Function），充当 GRPO 的硬截断开关。

#### b. NFT 的梯度形态

此时，NFT 引入了核心工程补丁——**提示词动态难度加权**，将其设为 $\omega(q) = \sqrt{(1-\hat{r}_q)/\hat{r}_q}$。对完全体工业级损失函数（公式 10）关于参数 $\theta$ 求导，神奇的事情发生了，其解析梯度表达为：

$$\nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{NFT}}(\theta) = - \sum_{q,a,r} \left\{ r A_q^+ \cdot \frac{1}{R_\theta^t(q,a)} + (1-r) A_q^- \cdot \max\left[\frac{1-\hat{r}_q R_\theta^t(q,a)}{1-\hat{r}_q}, \epsilon\right]^{-1} \right\} \nabla_\theta R_\theta^t(q,a)$$

---

### 2. 核心定理：Proposition 4.2 同策略绝对等价性

仔细凝视上述两个完全由不同范式推导出来的梯度公式，当训练处于完全同策略（Strictly On-Policy）阶段时，即新旧模型几乎完全一致、Token 级似然比 $R_\theta^t(q, a) \equiv 1$ 且截断超参 $\epsilon \le 1$ 时：

1. **正样本项（$r=1$）**：
* GRPO 项：由于 $R_\theta^t = 1 < 1+\epsilon'$ 恒成立，指示函数 $\mathcal{I} = 1$，梯度权重为 $A_q^+$。
* NFT 项：由于 $\frac{1}{R_\theta^t} = \frac{1}{1} = 1$，梯度权重同样缩并为 $A_q^+$。


2. **负样本项（$r=0$）**：
* GRPO 项：由于 $R_\theta^t = 1 > 1-\epsilon'$ 恒成立，指示函数 $\mathcal{I} = 1$，梯度权重为 $A_q^-$。
* NFT 项：将 $R_\theta^t = 1$ 代入括号内部：

$$\max\left[\frac{1-\hat{r}_q \cdot 1}{1-\hat{r}_q}, \epsilon\right]^{-1} = \max[1, \epsilon]^{-1} = 1^{-1} = 1$$



因此，NFT 的负样本梯度权重也完美缩并为 $A_q^-$。



由此，论文完成了里程碑式的理论证明（**Proposition 4.2**）：


$$\text{当 } R_\theta^t(q,a) = 1 \implies \nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{NFT}}(\theta) \equiv \nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{GRPO}}(\theta)$$

> **物理含义**：NFT 作为一个纯监督学习（SL）框架，**在同策略在线训练时，其梯度流与标准的强化学习（GRPO）完全同构**。它不需要 RL 的任何特殊算子，仅靠全概率公式的自发演推，就在宏观和微观两个层面同时复刻了 RL 的行为。

---

### 3. 分歧点深度剖析：硬截断 vs. 软 decay（Figure 4 解读）

既然同策略时完全等价，那么两者的 Gap 究竟在哪里？论文的一句话一针见血：**唯一的区别，在于当数据走向异策略（Off-Policy）时，两者对抗过拟合的截断策略不同。**

结合原著中的 **Figure 4**（梯度权重演变曲线），我们可以清晰地看到两者的分歧轨迹：

#### a. 对于正样本（$r=1$）

* **GRPO（绿线）**：实行**硬截断（Hard Clip）**。一旦当前模型把某个正样本的概率拉得太高，导致 $R_\theta^t > 1+\epsilon'$，GRPO 会瞬间把指示函数抹零（$\mathcal{I}=0$），**梯度直接断崖式归零**。这意味着“我知道它很好，但不要太贪婪，停止向这个方向更新”。
* **NFT（蓝线）**：实行**软衰减（Softer Decay Schedule）**。随着正样本概率的不断增大，$R_\theta^t$ 变大，其梯度权重以 $\frac{1}{R_\theta^t}$ 的动态倒数形式**平滑且无限地向 0 逼近**。

#### b. 对于负样本（$r=0$）

* **GRPO（绿线）**：同样是硬截断。一旦当前模型把错题 Token 的概率成功压低到 $R_\theta^t < 1-\epsilon'$ 以下，GRPO 的打压梯度瞬间清零，停止惩罚。
* **NFT（蓝线）**：展现出极为激进的反思路径。随着错题概率 $R_\theta^t$ 被不断打压（向 0 逼近），分子 $1 - \hat{r}_q R_\theta^t$ 会越来越接近 1。此时，整个倒数项 $\left(\frac{1 - \hat{r}_q R_\theta^t}{1 - \hat{r}_q}\right)^{-1}$ 反而会**平滑地一路上扬**，直到触及 $\epsilon^{-1}$ 的最高防爆阀限值。这表明 NFT 具有一种“斩草除根”的倾向，会以更软但更持续的跨度去清理错误路径的残余概率。

---

### 4. 隐式群体归一化（Implicit Group Normalization）的理论洗礼

这一节最后的一段话，把 NFT 的理论高度再次拉升了一个台阶。

在强化学习中，GRPO 的“群体归一化（Group Normalization）”最初只是一个纯经验主义（Empirical Technique）的工程 Trick——大家发现把一组样本的 Reward 做个减均值除方差的标准化，训练就会稳定。但为什么要这么做？背后的理论根基是什么？此前没人能说清。

而 Proposition 4.1 揭示了：**这个归一化优势项（$A_q^+$ 和 $A_q^-$）早已如同基因一般，隐式地自发存在于 NFT 全概率公式推导出的天然结构中。** * 当题目的经验正确率 $\hat{r}_q$ 极低时（硬核错题），正样本的优势值 $A_q^+ = \sqrt{(1-\hat{r}_q)/\hat{r}_q}$ 会变得极大。这意味着，NFT 会自动拨调巨量梯度去狠狠奖励那个罕见的做对该题的 Token。

* 反之，当正确率 $\hat{r}_q$ 极高时（简单题），偶尔做错的负样本优势值 $A_q^-$ 会变得极有分量，驱动模型用极大的推力去擦除这个意外的污点。

这为 GRPO 的群体归一化提供了完美的**第一性原理证明**。论文通过证明通过调整 $\omega(q) = 1 - \hat{r}_q$ 还可以进一步与 Dr. GRPO 算法完美对齐，正式宣告：**监督学习（SL）与强化学习（RL）两大框架在数学根源上，本就共享着同一套守恒的灵魂。**

---

## 四、 NFT 的工业级算法流程

在实际落地中，NFT 采用完全在线（Online）迭代的 Pipeline：

1. **同策略采样（Rollout）：** 当前策略 $\pi_\theta$ 对数学题库 $Q$ 进行批量采样，每个问题生成多个回答。
2. **二元验证（Evaluation）：** 动用验证器（Verifier）将生成的解答自动归类为正样本池 $\mathcal{D}^+$ 和负样本池 $\mathcal{D}^-$，并统计出每道题的当前胜率 $r_q$。
3. **计算梯度并更新（NFT Step）：**
* 正样本做标准的负对数似然（极大化正确 Token 概率）。
* 负样本通过全概率减法 Loss 做梯度上升（打压错误路径），且其惩罚力度自动受到 $\frac{r_q}{1-r_q}$ 的动态调节。


4. **同步循环：** 用更新后的模型替代老模型，进入下一轮 Rollout。

---

## 五、 总结与论文的启示

> 1. **极致的显存优势**：传统的 RL 需要 Actor, Critic, Reference, Reward 等多个网络。即使是省去 Critic 的 GRPO，也必须冻结一个 Reference 模型来计算 KL 散度。而 NFT 作为一个纯 SL 方法，在训练中**只需要维护一个当前正在优化的 Policy 模型**，极大地释放了显存和吞吐量。
> 2. **信息的全面压榨**：NFT 证明了错题不是垃圾，而是资产。通过减法结构强迫模型“向失败反思”。
> 3. **范式的完美统一**：它提供了一座数学桥梁，证明了看似截然不同的监督学习（几率最大化）与强化学习（奖励最大化），在自我提升的终点上其实是殊途同归的。
> 
> 

---

在这个彻底擦干杂质的全新全概率框架下，我们可以更敏锐地看到它的边界。你觉得当模型遇到完全不会的“死穴题”（$r_q \to 0$）或全对的“简单题”（$r_q \to 1$）时，这种纯线性减法在训练初始阶段会不会引发梯度爆炸或梯度消失的病态表现？

## 从**最底层的概率论与变分法**开始推，把所有的数学细节全部铺开。

---

## 一、从概率论第一原理推导“策略拆分”

大模型在生成文本时，本质上是一个条件概率分布。
我们定义三个严格的随机变量：

* $S$：输入的状态/问题空间（Prompt）。
* $A$：动作/输出文本空间（Response）。
* $Y$：二元评判标签空间，$Y \in \{0, 1\}$（$1$ 代表答案正确，$0$ 代表答案错误）。

大模型当前的整体策略定义为条件概率：$\pi_\theta(a|s) \triangleq P(A=a | S=s)$。

### 1. 应用全概率公式

在概率论中，对于任何随机变量，我们都可以引入一个互斥且完备的事件组（这里是 $Y=1$ 和 $Y=0$）来做边缘化展开：


$$P(A=a | S=s) = P(A=a, Y=1 | S=s) + P(A=a, Y=0 | S=s)$$

### 2. 应用条件概率乘法公式

根据概率论经典的乘法公式 $P(X, Y | Z) = P(Y | Z) \cdot P(X | Y, Z)$，我们将上式右边的两项联合概率分别展开：

1. **正样本项（做对的概率）**：

$$P(A=a, Y=1 | S=s) = P(Y=1 | S=s) \cdot P(A=a | Y=1, S=s)$$


2. **负样本项（做错的概率）**：

$$P(A=a, Y=0 | S=s) = P(Y=0 | S=s) \cdot P(A=a | Y=0, S=s)$$



### 3. 代入严格的隐式子策略定义

为了将概率论公式转化为大模型训练的语言，我们对展开后的四项做如下定义：

* **当前模型的胜率（正确率）**：记为 $\eta_\theta(s) \triangleq P(Y=1|S=s)$。因为 $Y$ 是二元的，所以模型做错的概率天然为 $P(Y=0|S=s) = 1 - \eta_\theta(s)$。
* **虚拟正策略 $\pi_\theta^+(a|s)$**：定义为在“确保能做对（$Y=1$）”的约束下，模型生成文本的条件概率：

$$\pi_\theta^+(a|s) \triangleq P(A=a | Y=1, S=s)$$


* **虚拟负策略 $\pi_\theta^-(a|s)$**：定义为在“注定会做错（$Y=0$）”的约束下，模型生成文本的条件概率：

$$\pi_\theta^-(a|s) \triangleq P(A=a | Y=0, S=s)$$



将这些符号放回步骤 1.1 的加法式子中，我们得到了毫无数学争议的**策略拆分恒等式（公式一）**：


$$\pi_\theta(a|s) = \eta_\theta(s) \cdot \pi_\theta^+(a|s) + (1 - \eta_\theta(s)) \cdot \pi_\theta^-(a|s)$$


---

## 二、 NFT 梯度的完整数学推导

我们从工业级损失函数（公式 10）出发。为了让推导聚焦，我们考察**单个问题 $q$、单个回答 $a$ 在单个 Token $t$ 上的损失项 $\mathcal{L}_{q,a,t}(\theta)$**：

$$\mathcal{L}_{q,a,t}(\theta) = - \omega(q) \left[ r \log R_\theta^t(q, a) + (1 - r) \log \max\left( \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right) \right]$$

我们需要求它对参数 $\theta$ 的偏导 $\nabla_\theta \mathcal{L}_{q,a,t}(\theta)$。根据二元指针 $r \in \{0, 1\}$，我们分正、负样本两种情况进行链式法则求导。

### 1. 正样本情况（$r = 1$）

当 $r=1$ 时，损失函数简化为：


$$\mathcal{L}^+_{q,a,t}(\theta) = - \omega(q) \log R_\theta^t(q, a)$$

根据复合函数求导法则（$\frac{d}{dx}\log x = \frac{1}{x}$）：


$$\nabla_\theta \mathcal{L}^+_{q,a,t}(\theta) = - \omega(q) \cdot \frac{1}{R_\theta^t(q, a)} \cdot \nabla_\theta R_\theta^t(q, a)$$

---

### 2. 负样本情况（$r = 0$）

当 $r=0$ 时，损失函数简化为：


$$\mathcal{L}^-_{q,a,t}(\theta) = - \omega(q) \log \max\left( \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right)$$

这里遇到了 `max` 算子。根据论文设计，作者引入了 **STE（直通估计器）**，在前向传播时做截断，在反向传播求导时，假定梯度直接穿透至内部的函数分支。

令内部函数分支为 $u(\theta) = \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}$，对其应用链式法则：


$$\nabla_\theta \mathcal{L}^-_{q,a,t}(\theta) = - \omega(q) \cdot \max[u(\theta), \epsilon]^{-1} \cdot \nabla_\theta u(\theta)$$

现在，我们对内部函数 $u(\theta)$ 单独关于 $\theta$ 求导：


$$\nabla_\theta u(\theta) = \nabla_\theta \left( \frac{1}{1 - \hat{r}_q} - \frac{\hat{r}_q}{1 - \hat{r}_q} R_\theta^t(q, a) \right) = - \frac{\hat{r}_q}{1 - \hat{r}_q} \nabla_\theta R_\theta^t(q, a)$$

将 $\nabla_\theta u(\theta)$ 的结果代回原式，**注意负负得正**：


$$\nabla_\theta \mathcal{L}^-_{q,a,t}(\theta) = - \omega(q) \cdot \max\left[ \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right]^{-1} \cdot \left( - \frac{\hat{r}_q}{1 - \hat{r}_q} \nabla_\theta R_\theta^t(q, a) \right)$$

$$\nabla_\theta \mathcal{L}^-_{q,a,t}(\theta) = + \left( \omega(q) \frac{\hat{r}_q}{1 - \hat{r}_q} \right) \cdot \max\left[ \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right]^{-1} \nabla_\theta R_\theta^t(q, a)$$

---

### 3. 梯度合并与统一表征

为了写成和论文 Proposition 4.1 一致的形式，我们将正负两项重新用 $r$ 和 $(1-r)$ 合并，并在最外层**强行提取一个负号 $-$** 和公共梯度项 $\nabla_\theta R_\theta^t(q, a)$：

$$\nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{NFT}}(\theta) = - \sum_{q,a,r} \sum_{t} \left\{ r \cdot \color{blue}{\omega(q)} \cdot \frac{1}{R_\theta^t(q, a)} + (1-r) \cdot \color{red}{\left( - \omega(q) \frac{\hat{r}_q}{1 - \hat{r}_q} \right)} \cdot \max\left[ \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right]^{-1} \right\} \nabla_\theta R_\theta^t(q, a)$$

---

### 4. 引入 $\omega(q)$ 代数消元：逼出 Advantage 归一化项

论文在此处亮出了核心杀招，将提示词难度权重定义为：


$$\omega(q) = \sqrt{\frac{1 - \hat{r}_q}{\hat{r}_q}}$$

现在，我们将这个特定的 $\omega(q)$ 分别代入上面大括号里的**蓝色**与**红色**系数项中：

* **蓝色系数项（正样本权重）：**

$$\omega(q) = \sqrt{\frac{1 - \hat{r}_q}{\hat{r}_q}} \equiv A_q^+$$



这正好就是 GRPO 里的正样本标准优势值！
* **红色系数项（负样本权重）：**

$$- \omega(q) \frac{\hat{r}_q}{1 - \hat{r}_q} = - \sqrt{\frac{1 - \hat{r}_q}{\hat{r}_q}} \cdot \frac{\hat{r}_q}{1 - \hat{r}_q}$$



我们将根号外的 $\frac{\hat{r}_q}{1 - \hat{r}_q}$ 强行塞进根号内部（平方后相乘）：

$$= - \sqrt{\frac{1 - \hat{r}_q}{\hat{r}_q} \cdot \frac{\hat{r}_q^2}{(1 - \hat{r}_q)^2}} = - \sqrt{\frac{\hat{r}_q}{1 - \hat{r}_q}} \equiv A_q^-$$



这又丝丝入扣地对齐了 GRPO 里的负样本标准优势值！

将化简后的 $A_q^+$ 和 $A_q^-$ 填回大括号，**严格导出论文命题 4.1(b) 的 NFT 最终梯度形态**：


$$\nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{NFT}}(\theta) = - \sum_{q,a,r} \sum_{t} \left\{ r A_q^+ \frac{1}{R_\theta^t(q, a)} + (1-r) A_q^- \max\left[ \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right]^{-1} \right\} \nabla_\theta R_\theta^t(q, a)$$

---

## 三、 NFT 与 GRPO 梯度流的深度对撞比对

现在我们将两个算法在二元奖励下的完整解析梯度写在上下两行，进行逐项扫描比对：

### 1. 梯度的宏观同构性

$$\nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{GRPO}}(\theta) = - \sum_{q,a,r} \sum_{t} \left\{ r \cdot A_q^+ \cdot \color{green}{\mathcal{I}[R_\theta^t < 1 + \epsilon']} + (1-r) \cdot A_q^- \cdot \color{green}{\mathcal{I}[R_\theta^t > 1 - \epsilon']} \right\} \nabla_\theta R_\theta^t(q,a)$$

$$\nabla_\theta \mathcal{L}_{\mathcal{D}}^{\text{NFT}}(\theta) = - \sum_{q,a,r} \sum_{t} \left\{ r \cdot A_q^+ \cdot \color{blue}{\frac{1}{R_\theta^t(q, a)}} + (1-r) \cdot A_q^- \cdot \color{blue}{\max\left[ \frac{1 - \hat{r}_q R_\theta^t(q, a)}{1 - \hat{r}_q}, \epsilon \right]^{-1}} \right\} \nabla_\theta R_\theta^t(q,a)$$

从结构上看，两者共享了完全相同的核心骨架：

* **相同的外部控制**：最外层的负号、样本与 Token 双重求和。
* **相同的基础梯度矢量**：$\nabla_\theta R_\theta^t(q,a)$ 指引了参数更新的基准方向。
* **相同的动态优势调节器**：$A_q^+$ 和 $A_q^-$ 负责在跨题目（难度不同）时自动平衡梯度权重。

唯一的交锋点，全部收敛在绿色项（GRPO）与蓝色项（NFT）的**截断/调节机制**上。

---

### 2. 异策略（Off-Policy）更新行为的微观差异

当模型经过几次 Step 的更新，当前策略 $\pi_\theta$ 开始偏离采样时的旧策略 $\pi_{\text{old}}$（即 $R_\theta^t \neq 1$）时，两者的行为表现出截然不同的工程哲学：

#### 🥊 正样本更新（$r=1$）

* **GRPO（硬开关）**：当 $R_\theta^t$ 增长到超过 $1+\epsilon'$ 时，说明当前模型已经比旧模型好太多了。GRPO 触发机制，指示函数 $\mathcal{I}$ 变为 0，该 Token 梯度**直接归零，强行刹车**。
* **NFT（软自适应）**：随着 $R_\theta^t$ 的膨胀，其权重项 $\frac{1}{R_\theta^t}$ 开始**平滑平流衰减**。模型概率拉得越高，后续的拉力就越温柔，而不是像 GRPO 那样一刀切。

#### 🥊 负样本打压（$r=0$）

* **GRPO（硬开关）**：一旦模型成功把错题 Token 的概率踩下去，导致 $R_\theta^t < 1-\epsilon'$，GRPO 认为惩罚已经足够，梯度**瞬间清零，放过该 Token**。
* **NFT（追击反思）**：随着错题 Token 的概率被不断打压（$R_\theta^t \to 0$），分子 $1 - \hat{r}_q R_\theta^t$ 越来越接近 1。此时整个倒数项：

$$\left( \frac{1 - \hat{r}_q R_\theta^t}{1 - \hat{r}_q} \right)^{-1} \to \frac{1 - \hat{r}_q}{1} = 1 - \hat{r}_q$$



这意味着即使错题概率降得很低，NFT 依然会保持一个稳定的基础权重 $(1-\hat{r}_q) A_q^-$ 去持续清理错误路径的残余概率，直到逼近防爆上限 $\epsilon^{-1}$。

### 3. 终点复归：On-Policy 下的数学融合

当模型处于完全同策略状态（$R_\theta^t \equiv 1$）时，将 $R_\theta^t = 1$ 代入：

* GRPO 的两个指示函数全为 1。
* NFT 的正样本项 $\frac{1}{1} = 1$；负样本项 $\max[1, \epsilon]^{-1} = 1$。

两套公式的大括号内部完全缩并为：


$$\{ r A_q^+ + (1-r) A_q^- \}$$

至此，论文在数学上完成了最漂亮的一击：**NFT 的纯监督学习梯度在起点和同策略状态下，与 GRPO 的强化学习策略梯度完全重合。** 所谓的强化学习优势归一化，在监督学习的全概率守恒求导中，只是一个自发浮出水面的代数必然。